<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Docker Volumes, Bind Mounts, and Cleanup

Containers need storage beyond their writable layer when data must outlast one container. This notebook introduces named volumes, bind mounts, and cleanup of local Docker resources.

## Persist one file in a named volume

Run these commands only against the authorized local base-station Docker daemon and use only the supplied `lx-docker-*` resource names; do not substitute an unfamiliar resource or remote target.

Container writable layers disappear when a `--rm` container is removed. A __named volume__ stores selected data outside that container lifecycle. Create a learner-owned volume, write one note, and read it back through a new container:

```bash
docker volume create lx-docker-notes
docker run --rm --mount type=volume,src=lx-docker-notes,dst=/notes \
  alpine:3.21 sh -c 'printf "%s\n" "Docker data persists in a named volume." > /notes/note.txt'
docker run --rm --mount type=volume,src=lx-docker-notes,dst=/notes,readonly \
  alpine:3.21 cat /notes/note.txt
```

The second container can read `note.txt` even though the first container no longer exists. `--mount` states the mount type, source, and destination explicitly. `readonly` prevents the reading container from changing the volume. A volume has its own lifecycle, so remove it explicitly when this exercise is complete.

## Inspect a read-only bind mount

From the LX repository root, save the supplied exercise path for the read-only bind-mount example:

```bash
LX_ROOT="$PWD"
```

A __bind mount__ exposes a directory from the base station filesystem to a container. Unlike a named volume, its source is a path you choose. Inspect the exercise directory through a read-only bind mount:

```bash
docker run --rm \
  --mount type=bind,src="$LX_ROOT"/packages/docker_exercises,dst=/exercise,readonly \
  alpine:3.21 ls -la /exercise
```

The container sees the same exercise files that are stored on the base station. A writable bind mount would let the container change host files, so this LX uses `readonly`. Do not bind-mount sensitive directories, home directories, or the Docker socket into practice containers.

## Clean up only LX resources

Stop and remove the named web container, remove the named volume, and remove the image built by this notebook:

```bash
docker stop lx-docker-web
docker container rm lx-docker-web
docker volume rm lx-docker-notes
docker image rm lx-docker-hello:0.1
```

`docker stop` requests a clean shutdown of the web server. `docker container rm` then removes that exact stopped container. The small base images remain cached because they might be used elsewhere. Each cleanup command names one resource created by this LX; do not substitute a broad prune command.

## Further reading

Docker's official guides explain [persistent data](https://docs.docker.com/get-started/docker-concepts/running-containers/persisting-container-data/) and [bind mounts](https://docs.docker.com/engine/storage/bind-mounts/).

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
